In [ ]:
# !pip install ninja --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# !pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# !pip install compressed-tensors --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.13.0+cu130
CUDA Available: True
CUDA Version: 13.0
GPU Name: NVIDIA H200 NVL
VRAM: 139.8 GB


In [4]:
MODEL_ID = "Qwen/Qwen3.5-9B"

OUTPUT_BASE_DIR = "./Qwen3.5-9B"
LOCAL_PATH = "./local_model_Qwen3.5-9B"

In [5]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Hint: A new version of huggingface_hub (1.29.0) is available! You are using version 1.27.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 16 files:   0%|                                | 0/16 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|       |  0.00B / 11.5kB            
Reconstructing (incomplete total...):   0%|       | 11.5kB / 5.37GB            Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0%|       | 11.5kB / 5.37GB            

Fetching 16 files:  12%|███                     | 2/16 [00:00<00:00, 18.67it/s]Still waiting to acquire lock on /workspace/local_model_Qwen3.5-9B/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting 

In [6]:
from safetensors import safe_open
import os

for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors-00003-of-00004.safetensors
mtp.fc.weight
mtp.layers.0.self_attn.q_proj.weight

Checking model.safetensors-00002-of-00004.safetensors
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight

Checking model.safetensors-00001-of-00004.safetensors

Checking model.safetensors-00004-of-00004.safetensors
mtp.layers.0.input_layernorm.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

In [8]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
         

In [9]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [10]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 800,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 1,  # Faster on 48GB VRAM
    "seqlen": 2048*2,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    "layer_config": {
        "mtp": {"data_type": "bfloat16"},
        "mtp.fc": {"data_type": "bfloat16"}
    }
}

In [11]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-09-01 05:01:58 WARNING autoround.py L591: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-09-01 05:01:58 WARNING autoround.py L591: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-09-01 05:01:58 WARNING autoround.py L591: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [12]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,llm_compressor", inplace=True
)

2026-09-01 05:01:58 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.visual.blocks.[0-26].mlp.linear_fc1, model.visual.blocks.[0-26].mlp.linear_fc2
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-09-01 05:01:59 INFO base.py L1294: `torch.compile` is enabled
2026-09-01 05:01:59 INFO orchestrator.py L570: start to cache block inputs
2026-09-01 05:01:59 INFO mllm.py L86: Using MLLM template: qwen3_5
2026-09-01 05:01:59 INFO calib_dataset.py L1113: Preprocessing calibration dataset in a subprocess to avoid memory leaks...


README.md:   0%|          | 0.00/373 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/921 [00:00<?, ?B/s]

data/train-00000-of-00001-4746b8785c874c(…): reconstructing file:   0%|          |  0.00B / 33.3MB            

data/train-00000-of-00001-4746b8785c874c(…): downloading bytes:           |  0.00B            

...

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-09-01 05:13:48 INFO missing_tensors.py L373: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 3/3 [00:00<00:00, 426.08shard/s]
2026-09-01 05:13:49 INFO missing_tensors.py L513: Successfully wrote 15 missing tensor(s) to 'model_extra_tensors.safetensors' in ./Qwen3.5-9B/local_model_Qwen3.5-9B-w4g32/auto-gptq.


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-09-01 05:14:33 INFO missing_tensors.py L373: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors: 100%|██████████| 3/3 [00:00<00:00, 327.04shard/s]
2026-09-01 05:14:34 INFO missing_tensors.py L513: Successfully wrote 15 missing tensor(s) to 'model_extra_tensors.safetensors' in ./Qwen3.5-9B/local_model_Qwen3.5-9B-w4g32/llm-compressor-wint-a16.
2026-09-01 05:14:34 INFO device.py L1448: 'peak_ram': 35.48GB, 'peak_vram': 50.44GB


(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1152)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-26): 27 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1152, out_features=3456, bias=True)
             (proj): Linear(in_features=1152, out_features=1152, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
             (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
             (act_fn): 

In [16]:
HF_USER = "J-Fraudster"

In [ ]:
base_name = MODEL_ID.split("/")[-1]
hf_token = "hf_FmyxxxxxxxxxxxxxxxxxxxxxxpaiQ"

In [ ]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model_Qwen3.5-9B-w4g32/auto-round-auto-gptq"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model_Qwen3.5-9B-w4g32/auto-gptq"), 
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
    push_to_hub(
            os.path.join(OUTPUT_BASE_DIR, "local_model_Qwen3.5-9B-w4g32/llm-compressor-wint-a16"), 
            f"{base_name}-W4A16-AutoRound-LLM-Compressor", 
            hf_token)
else:
    print("No Hugging Face token found. Skipping upload to hub.")